# CAMELS-RU Data Analysis Notebook

## Overview

This notebook performs comprehensive analysis of the CAMELS-RU dataset, including:

1. **Watershed Size Distribution** - Categorization and visualization of catchment areas
2. **Discharge Data Analysis** - Quality assessment, temporal coverage, and hydrological characteristics
3. **Water Level Data Analysis** - Stage measurement analysis and quality distribution
4. **Data Coverage Analysis** - Multi-variable integration and completeness metrics
5. **Report Generation** - Comprehensive markdown report for paper drafts

## Outputs

### Figures (saved to `paper/images/`)
- `watershed_size_distribution.png`
- `discharge_characteristics.png`
- `water_level_characteristics.png`
- `data_coverage_summary.png`

### Data Files (saved to `docs/analysis_results/`)
- `camels_ru_hydrological_analysis_report.md` - Main comprehensive report
- `watershed_size_distribution.csv`
- `watershed_area_statistics_by_category.csv`
- `data_coverage_summary.csv`
- `discharge_statistics_summary.csv`
- `water_level_statistics_summary.csv`
- `analysis_metadata.csv`

## Usage

Run all cells sequentially. The analysis samples discharge and level data for efficiency (100 decent, 50 poor discharge files; 100 decent, 20 poor level files). To analyze the full dataset, modify the slicing in cells 7 and 11.

## Requirements

- Python 3.12
- geopandas, pandas, matplotlib, numpy
- Data files in `../data/HydroFiles/` and `../data/Geometry/`

---

In [ ]:
"""CAMELS-RU Data Analysis Notebook.

Comprehensive analysis of hydrological observations and catchment characteristics.
"""
import sys
import warnings
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.append("../")
from src.static.hydro_atlas_analysis import (
    categorize_catchment_size,
    get_size_categories,
)
from src.utils.logger import setup_logger

logger = setup_logger("paperbook_analysis", logger_name="../logs/paperbook_analysis.log")

gpd.options.io_engine = "pyogrio"
warnings.simplefilter(action="ignore", category=FutureWarning)

plt.rcParams["font.family"] = "DeJavu Serif"
plt.rcParams["font.serif"] = ["Times New Roman"]
plt.rcParams["figure.dpi"] = 100
plt.rcParams["savefig.dpi"] = 300

# Define paths
BASE_DIR = Path("../")
DATA_DIR = BASE_DIR / "data"
HYDRO_DIR = DATA_DIR / "HydroFiles"
GEOM_DIR = DATA_DIR / "Geometry"
OUTPUT_DIR = BASE_DIR / "docs" / "analysis_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

logger.info("Notebook initialized successfully")

In [ ]:
# Load spatial data
logger.info("Loading spatial data...")
ws = gpd.read_file(GEOM_DIR / "WatershedGeomCAMELS.gpkg")
ws.set_index("gauge_id", inplace=True)

gauge = gpd.read_file(GEOM_DIR / "GaugeGeomCAMELS.gpkg")
gauge.set_index("gauge_id", inplace=True)

basemap_data = gpd.read_file(GEOM_DIR / "basemap.gpkg")

logger.info(f"Loaded {len(ws)} watersheds and {len(gauge)} gauges")
print(f"Total watersheds: {len(ws)}")
print(f"Total gauges: {len(gauge)}")
print(f"\nWatershed area statistics (km²):")
print(ws["area"].describe())

## 1. Watershed Size Distribution Analysis

In [ ]:
# Categorize watersheds by size
ws["size_category"] = ws["area"].apply(categorize_catchment_size)
ws["size_category"] = pd.Categorical(ws["size_category"], categories=get_size_categories())

# Compute statistics
size_counts = ws["size_category"].value_counts(sort=False)
total_gauges = len(ws)

# Create summary DataFrame
size_summary = pd.DataFrame(
    {
        "Size Category": size_counts.index,
        "Count": size_counts.values,
        "Percentage": (size_counts.values / total_gauges * 100).round(2),
    }
)

# Calculate area statistics per category
area_stats = []
for cat in get_size_categories():
    cat_data = ws[ws["size_category"] == cat]["area"]
    area_stats.append(
        {
            "Category": cat,
            "Min (km²)": cat_data.min(),
            "Mean (km²)": cat_data.mean(),
            "Median (km²)": cat_data.median(),
            "Max (km²)": cat_data.max(),
        }
    )
area_stats_df = pd.DataFrame(area_stats)

print("=== Watershed Size Distribution ===\n")
print(size_summary.to_string(index=False))
print("\n=== Area Statistics by Category ===\n")
print(area_stats_df.to_string(index=False))

# Plot distribution
fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.bar(
    range(len(size_counts)),
    size_counts.values,
    color="steelblue",
    edgecolor="black",
    linewidth=1.2,
)
ax.set_xticks(range(len(size_counts)))
ax.set_xticklabels([cat.split(") ")[1] for cat in size_counts.index], rotation=30, ha="right")
ax.set_xlabel("Catchment Size Category", fontsize=14, fontweight="bold")
ax.set_ylabel("Number of Gauges", fontsize=14, fontweight="bold")
ax.set_title(
    "Distribution of CAMELS-RU Catchments by Size",
    fontsize=16,
    fontweight="bold",
    pad=20,
)
ax.grid(axis="y", alpha=0.3, linestyle="--")

# Add value labels
for i, (rect, cnt) in enumerate(zip(bars, size_counts.values, strict=False)):
    pct = cnt / total_gauges * 100
    ax.text(
        rect.get_x() + rect.get_width() / 2,
        rect.get_height() + total_gauges * 0.01,
        f"{cnt}\n({pct:.1f}%)",
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
    )

fig.tight_layout()
fig.savefig("../paper/images/watershed_size_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

logger.info("Watershed size distribution analysis complete")

## 2. Discharge Data Analysis

In [ ]:
# Scan discharge data files across all categories
logger.info("Scanning discharge data across all categories...")

# Define discharge data categories
discharge_categories = {
    "full_decent": HYDRO_DIR / "Discharge" / "full" / "decent",
    "full_poor": HYDRO_DIR / "Discharge" / "full" / "poor",
    "partial_decent": HYDRO_DIR / "Discharge" / "partial" / "decent",
    "partial_poor": HYDRO_DIR / "Discharge" / "partial" / "poor",
}

# Scan all categories
discharge_data = {}
discharge_ids_by_cat = {}
for cat_name, cat_path in discharge_categories.items():
    files = list(cat_path.glob("*.csv"))
    discharge_data[cat_name] = files
    discharge_ids_by_cat[cat_name] = {f.stem for f in files}
    print(f"Discharge {cat_name.replace('_', ' ')}: {len(files)} files")

# Aggregate by completeness type
full_decent_ids = discharge_ids_by_cat["full_decent"]
full_poor_ids = discharge_ids_by_cat["full_poor"]
partial_decent_ids = discharge_ids_by_cat["partial_decent"]
partial_poor_ids = discharge_ids_by_cat["partial_poor"]

all_full_ids = full_decent_ids | full_poor_ids
all_partial_ids = partial_decent_ids | partial_poor_ids
all_discharge_ids = all_full_ids | all_partial_ids

print(f"\n=== Discharge Data Summary ===")
print(f"Full records: {len(all_full_ids)} ({len(full_decent_ids)} decent, {len(full_poor_ids)} poor)")
print(
    f"Partial records: {len(all_partial_ids)} ({len(partial_decent_ids)} decent, {len(partial_poor_ids)} poor)"
)
print(f"Total unique gauges: {len(all_discharge_ids)}")

# Check coverage
ws_ids = set(ws.index)
discharge_coverage = len(all_discharge_ids & ws_ids)
print(
    f"\nSpatial coverage: {discharge_coverage}/{len(ws_ids)} ({discharge_coverage / len(ws_ids) * 100:.1f}%)"
)

In [ ]:
# Analyze discharge time series characteristics across all categories
logger.info("Analyzing discharge time series across all categories...")


def analyze_timeseries(file_path: Path, category: str, quality: str) -> dict:
    """Extract time series statistics from a discharge file."""
    try:
        df = pd.read_csv(file_path, parse_dates=["date"])
        return {
            "gauge_id": file_path.stem,
            "category": category,
            "quality": quality,
            "start_date": df["date"].min(),
            "end_date": df["date"].max(),
            "n_records": len(df),
            "n_years": (df["date"].max() - df["date"].min()).days / 365.25,
            "mean_q_cms": df["q_cms"].mean(),
            "median_q_cms": df["q_cms"].median(),
            "std_q_cms": df["q_cms"].std(),
            "min_q_cms": df["q_cms"].min(),
            "max_q_cms": df["q_cms"].max(),
            "q_cv": df["q_cms"].std() / df["q_cms"].mean() if df["q_cms"].mean() > 0 else np.nan,
            "missing_pct": (df["q_cms"].isna().sum() / len(df) * 100),
        }
    except Exception as e:
        logger.error(f"Error processing {file_path}: {e}")
        return None


# Process samples from each category
print("Processing discharge time series (sampling from each category)...")
discharge_stats = []

sample_sizes = {
    "full_decent": 80,
    "full_poor": 20,
    "partial_decent": 60,
    "partial_poor": 40,
}

for cat_name, sample_size in sample_sizes.items():
    files = discharge_data[cat_name][:sample_size]
    completeness, quality = cat_name.split("_")
    for f in files:
        stats = analyze_timeseries(f, completeness, quality)
        if stats:
            discharge_stats.append(stats)

discharge_df = pd.DataFrame(discharge_stats)
discharge_df.set_index("gauge_id", inplace=True)

print(f"\nProcessed {len(discharge_df)} discharge time series (sampled)")
print("\n=== Discharge Statistics by Category ===")
print(
    discharge_df.groupby(["category", "quality"])[["n_years", "mean_q_cms", "q_cv", "missing_pct"]].agg(
        ["count", "mean", "median"]
    )
)

In [ ]:
# Visualize discharge data characteristics by category
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Record length distribution by category
for cat in ["full", "partial"]:
    cat_data = discharge_df[discharge_df["category"] == cat]
    axes[0, 0].hist(cat_data["n_years"], bins=20, alpha=0.6, label=cat.capitalize(), edgecolor="black")
axes[0, 0].set_xlabel("Record Length (years)", fontweight="bold")
axes[0, 0].set_ylabel("Frequency", fontweight="bold")
axes[0, 0].set_title("Record Length Distribution", fontweight="bold")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Mean discharge distribution by category (log scale)
for cat in ["full", "partial"]:
    cat_data = discharge_df[discharge_df["category"] == cat]
    axes[0, 1].hist(
        np.log10(cat_data["mean_q_cms"] + 0.1),
        bins=20,
        alpha=0.6,
        label=cat.capitalize(),
        edgecolor="black",
    )
axes[0, 1].set_xlabel("log₁₀(Mean Discharge, m³/s)", fontweight="bold")
axes[0, 1].set_ylabel("Frequency", fontweight="bold")
axes[0, 1].set_title("Mean Discharge Distribution", fontweight="bold")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Coefficient of variation by category
for cat in ["full", "partial"]:
    cat_data = discharge_df[discharge_df["category"] == cat]
    axes[0, 2].hist(
        cat_data["q_cv"].dropna(), bins=20, alpha=0.6, label=cat.capitalize(), edgecolor="black"
    )
axes[0, 2].set_xlabel("Coefficient of Variation", fontweight="bold")
axes[0, 2].set_ylabel("Frequency", fontweight="bold")
axes[0, 2].set_title("Discharge Variability", fontweight="bold")
axes[0, 2].legend()
axes[0, 2].grid(alpha=0.3)

# Category distribution
cat_counts = discharge_df["category"].value_counts()
axes[1, 0].bar(cat_counts.index, cat_counts.values, color=["steelblue", "coral"], edgecolor="black")
axes[1, 0].set_xlabel("Completeness", fontweight="bold")
axes[1, 0].set_ylabel("Sample Count", fontweight="bold")
axes[1, 0].set_title("Full vs Partial Records", fontweight="bold")
axes[1, 0].grid(alpha=0.3, axis="y")

# Quality distribution
quality_counts = discharge_df["quality"].value_counts()
axes[1, 1].bar(
    quality_counts.index, quality_counts.values, color=["forestgreen", "orange"], edgecolor="black"
)
axes[1, 1].set_xlabel("Data Quality", fontweight="bold")
axes[1, 1].set_ylabel("Sample Count", fontweight="bold")
axes[1, 1].set_title("Quality Distribution", fontweight="bold")
axes[1, 1].grid(alpha=0.3, axis="y")

# Category × Quality matrix
cat_qual = discharge_df.groupby(["category", "quality"]).size().unstack(fill_value=0)
cat_qual.plot(kind="bar", ax=axes[1, 2], color=["forestgreen", "orange"], edgecolor="black")
axes[1, 2].set_xlabel("Completeness", fontweight="bold")
axes[1, 2].set_ylabel("Sample Count", fontweight="bold")
axes[1, 2].set_title("Category × Quality", fontweight="bold")
axes[1, 2].legend(title="Quality")
axes[1, 2].grid(alpha=0.3, axis="y")
axes[1, 2].tick_params(axis="x", rotation=0)

fig.tight_layout()
fig.savefig("../paper/images/discharge_characteristics.png", dpi=300, bbox_inches="tight")
plt.show()

logger.info("Discharge visualization complete")

## 3. Water Level Data Analysis

In [ ]:
# Scan water level data files across all categories
logger.info("Scanning water level data across all categories...")

# Define water level data categories
levels_categories = {
    "full_decent": HYDRO_DIR / "Levels" / "full" / "decent",
    "full_poor": HYDRO_DIR / "Levels" / "full" / "poor",
    "partial_decent": HYDRO_DIR / "Levels" / "partial" / "decent",
    "partial_poor": HYDRO_DIR / "Levels" / "partial" / "poor",
    "freezing_decent": HYDRO_DIR / "Levels" / "freezing" / "decent",
    "freezing_poor": HYDRO_DIR / "Levels" / "freezing" / "poor",
    "negatives_decent": HYDRO_DIR / "Levels" / "negatives" / "decent",
    "negatives_poor": HYDRO_DIR / "Levels" / "negatives" / "poor",
}

# Scan all categories
levels_data = {}
levels_ids_by_cat = {}
for cat_name, cat_path in levels_categories.items():
    files = list(cat_path.glob("*.csv"))
    levels_data[cat_name] = files
    levels_ids_by_cat[cat_name] = {f.stem for f in files}
    condition, quality = cat_name.rsplit("_", 1)
    print(f"Levels {condition} ({quality}): {len(files)} files")

# Aggregate by condition type
full_ids = levels_ids_by_cat["full_decent"] | levels_ids_by_cat["full_poor"]
partial_ids = levels_ids_by_cat["partial_decent"] | levels_ids_by_cat["partial_poor"]
freezing_ids = levels_ids_by_cat["freezing_decent"] | levels_ids_by_cat["freezing_poor"]
negatives_ids = levels_ids_by_cat["negatives_decent"] | levels_ids_by_cat["negatives_poor"]
all_levels_ids = full_ids | partial_ids | freezing_ids | negatives_ids

print(f"\n=== Water Level Data Summary ===")
print(f"Full records: {len(full_ids)}")
print(f"Partial records: {len(partial_ids)}")
print(f"Freezing issues: {len(freezing_ids)}")
print(f"Negative values: {len(negatives_ids)}")
print(f"Total unique gauges: {len(all_levels_ids)}")

# Check coverage and overlap
levels_coverage = len(all_levels_ids & ws_ids)
both_data = all_discharge_ids & all_levels_ids
print(
    f"\nSpatial coverage: {levels_coverage}/{len(ws_ids)} ({levels_coverage / len(ws_ids) * 100:.1f}%)"
)
print(
    f"Gauges with both Q & H: {len(both_data)} ({len(both_data) / len(all_discharge_ids) * 100:.1f}% of discharge gauges)"
)

In [ ]:
# Analyze water level time series characteristics across all categories
logger.info("Analyzing water level time series across all categories...")


def analyze_level_timeseries(file_path: Path, condition: str, quality: str) -> dict:
    """Extract time series statistics from a water level file."""
    try:
        df = pd.read_csv(file_path, parse_dates=["date"])
        return {
            "gauge_id": file_path.stem,
            "condition": condition,
            "quality": quality,
            "start_date": df["date"].min(),
            "end_date": df["date"].max(),
            "n_records": len(df),
            "n_years": (df["date"].max() - df["date"].min()).days / 365.25,
            "mean_lvl_sm": df["lvl_sm"].mean(),
            "median_lvl_sm": df["lvl_sm"].median(),
            "std_lvl_sm": df["lvl_sm"].std(),
            "min_lvl_sm": df["lvl_sm"].min(),
            "max_lvl_sm": df["lvl_sm"].max(),
            "lvl_range": df["lvl_sm"].max() - df["lvl_sm"].min(),
            "missing_pct": (df["lvl_sm"].isna().sum() / len(df) * 100),
            "negative_count": (df["lvl_sm"] < 0).sum() if "negatives" not in condition else None,
        }
    except Exception as e:
        logger.error(f"Error processing {file_path}: {e}")
        return None


# Process samples from each category
print("Processing water level time series (sampling from each category)...")
levels_stats = []

level_sample_sizes = {
    "full_decent": 60,
    "full_poor": 10,
    "partial_decent": 40,
    "partial_poor": 15,
    "freezing_decent": 25,
    "freezing_poor": 10,
    "negatives_decent": 30,
    "negatives_poor": 3,
}

for cat_name, sample_size in level_sample_sizes.items():
    files = levels_data[cat_name][:sample_size]
    condition, quality = cat_name.rsplit("_", 1)
    for f in files:
        stats = analyze_level_timeseries(f, condition, quality)
        if stats:
            levels_stats.append(stats)

levels_df = pd.DataFrame(levels_stats)
levels_df.set_index("gauge_id", inplace=True)

print(f"\nProcessed {len(levels_df)} water level time series (sampled)")
print("\n=== Water Level Statistics by Condition ===")
print(
    levels_df.groupby(["condition", "quality"])[
        ["n_years", "mean_lvl_sm", "lvl_range", "missing_pct"]
    ].agg(["count", "mean", "median"])
)

In [ ]:
# Visualize water level data characteristics by condition
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Record length distribution by condition
colors_cond = {"full": "steelblue", "partial": "coral", "freezing": "lightblue", "negatives": "salmon"}
for cond in ["full", "partial", "freezing", "negatives"]:
    cond_data = levels_df[levels_df["condition"] == cond]
    if len(cond_data) > 0:
        axes[0, 0].hist(
            cond_data["n_years"],
            bins=15,
            alpha=0.5,
            label=cond.capitalize(),
            color=colors_cond[cond],
            edgecolor="black",
        )
axes[0, 0].set_xlabel("Record Length (years)", fontweight="bold")
axes[0, 0].set_ylabel("Frequency", fontweight="bold")
axes[0, 0].set_title("Record Length by Condition", fontweight="bold")
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Mean level distribution by condition
for cond in ["full", "partial", "freezing", "negatives"]:
    cond_data = levels_df[levels_df["condition"] == cond]
    if len(cond_data) > 0:
        axes[0, 1].hist(
            cond_data["mean_lvl_sm"],
            bins=15,
            alpha=0.5,
            label=cond.capitalize(),
            color=colors_cond[cond],
            edgecolor="black",
        )
axes[0, 1].set_xlabel("Mean Water Level (cm)", fontweight="bold")
axes[0, 1].set_ylabel("Frequency", fontweight="bold")
axes[0, 1].set_title("Mean Level by Condition", fontweight="bold")
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Level range by condition
for cond in ["full", "partial", "freezing", "negatives"]:
    cond_data = levels_df[levels_df["condition"] == cond]
    if len(cond_data) > 0:
        axes[0, 2].hist(
            cond_data["lvl_range"],
            bins=15,
            alpha=0.5,
            label=cond.capitalize(),
            color=colors_cond[cond],
            edgecolor="black",
        )
axes[0, 2].set_xlabel("Level Range (cm)", fontweight="bold")
axes[0, 2].set_ylabel("Frequency", fontweight="bold")
axes[0, 2].set_title("Level Variability by Condition", fontweight="bold")
axes[0, 2].legend()
axes[0, 2].grid(alpha=0.3)

# Condition distribution
cond_counts = levels_df["condition"].value_counts()
axes[1, 0].bar(
    range(len(cond_counts)),
    cond_counts.values,
    color=[colors_cond[c] for c in cond_counts.index],
    edgecolor="black",
)
axes[1, 0].set_xticks(range(len(cond_counts)))
axes[1, 0].set_xticklabels(cond_counts.index, rotation=30, ha="right")
axes[1, 0].set_xlabel("Condition Type", fontweight="bold")
axes[1, 0].set_ylabel("Sample Count", fontweight="bold")
axes[1, 0].set_title("Records by Condition", fontweight="bold")
axes[1, 0].grid(alpha=0.3, axis="y")

# Quality distribution
quality_counts = levels_df["quality"].value_counts()
axes[1, 1].bar(
    quality_counts.index, quality_counts.values, color=["forestgreen", "orange"], edgecolor="black"
)
axes[1, 1].set_xlabel("Data Quality", fontweight="bold")
axes[1, 1].set_ylabel("Sample Count", fontweight="bold")
axes[1, 1].set_title("Quality Distribution", fontweight="bold")
axes[1, 1].grid(alpha=0.3, axis="y")

# Condition × Quality matrix
cond_qual = levels_df.groupby(["condition", "quality"]).size().unstack(fill_value=0)
cond_qual.plot(kind="bar", ax=axes[1, 2], color=["forestgreen", "orange"], edgecolor="black")
axes[1, 2].set_xlabel("Condition", fontweight="bold")
axes[1, 2].set_ylabel("Sample Count", fontweight="bold")
axes[1, 2].set_title("Condition × Quality", fontweight="bold")
axes[1, 2].legend(title="Quality")
axes[1, 2].grid(alpha=0.3, axis="y")
axes[1, 2].tick_params(axis="x", rotation=30)

fig.tight_layout()
fig.savefig("../paper/images/water_level_characteristics.png", dpi=300, bbox_inches="tight")
plt.show()

logger.info("Water level visualization complete")

## 4. Data Coverage and Completeness Analysis

In [ ]:
# Create comprehensive data coverage summary across all categories
logger.info("Generating comprehensive data coverage summary...")

coverage_summary = {
    "Total Watersheds": len(ws),
    "Total Gauges": len(gauge),
    "--- Discharge Data ---": "",
    "Full (Decent)": len(full_decent_ids),
    "Full (Poor)": len(full_poor_ids),
    "Partial (Decent)": len(partial_decent_ids),
    "Partial (Poor)": len(partial_poor_ids),
    "All Full Records": len(all_full_ids),
    "All Partial Records": len(all_partial_ids),
    "Total Discharge": len(all_discharge_ids),
    "--- Water Level Data ---": "",
    "Full (Decent)": len(levels_ids_by_cat["full_decent"]),
    "Full (Poor)": len(levels_ids_by_cat["full_poor"]),
    "Partial (Decent)": len(levels_ids_by_cat["partial_decent"]),
    "Partial (Poor)": len(levels_ids_by_cat["partial_poor"]),
    "Freezing (Decent)": len(levels_ids_by_cat["freezing_decent"]),
    "Freezing (Poor)": len(levels_ids_by_cat["freezing_poor"]),
    "Negatives (Decent)": len(levels_ids_by_cat["negatives_decent"]),
    "Negatives (Poor)": len(levels_ids_by_cat["negatives_poor"]),
    "All Full Levels": len(full_ids),
    "All Partial Levels": len(partial_ids),
    "Freezing Issues": len(freezing_ids),
    "Negative Values": len(negatives_ids),
    "Total Levels": len(all_levels_ids),
    "--- Integration ---": "",
    "Both Q & H": len(both_data),
    "Only Discharge": len(all_discharge_ids - all_levels_ids),
    "Only Levels": len(all_levels_ids - all_discharge_ids),
}

coverage_df = pd.DataFrame(
    [(k, v) for k, v in coverage_summary.items() if v != ""], columns=["Metric", "Count"]
)

print("=== CAMELS-RU Comprehensive Data Coverage ===\n")
print(coverage_df.to_string(index=False))

# Create enhanced visualization
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 3, hspace=0.3, wspace=0.3)

# Discharge data breakdown
ax1 = fig.add_subplot(gs[0, 0])
discharge_cats = ["Full\nDecent", "Full\nPoor", "Partial\nDecent", "Partial\nPoor"]
discharge_vals = [
    len(full_decent_ids),
    len(full_poor_ids),
    len(partial_decent_ids),
    len(partial_poor_ids),
]
bars1 = ax1.bar(
    discharge_cats,
    discharge_vals,
    color=["steelblue", "lightsteelblue", "coral", "lightsalmon"],
    edgecolor="black",
)
ax1.set_ylabel("Number of Gauges", fontweight="bold")
ax1.set_title("Discharge Data Categories", fontweight="bold", fontsize=12)
ax1.grid(alpha=0.3, axis="y")
for bar, val in zip(bars1, discharge_vals, strict=False):
    ax1.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(discharge_vals) * 0.01,
        str(val),
        ha="center",
        fontweight="bold",
        fontsize=9,
    )

# Water level data breakdown
ax2 = fig.add_subplot(gs[0, 1])
level_cats = [
    "Full\nDecent",
    "Full\nPoor",
    "Partial\nDecent",
    "Partial\nPoor",
    "Freezing\nDecent",
    "Freezing\nPoor",
    "Neg.\nDecent",
    "Neg.\nPoor",
]
level_vals = [
    len(levels_ids_by_cat[f"{c}_{q}"])
    for c, q in [
        ("full", "decent"),
        ("full", "poor"),
        ("partial", "decent"),
        ("partial", "poor"),
        ("freezing", "decent"),
        ("freezing", "poor"),
        ("negatives", "decent"),
        ("negatives", "poor"),
    ]
]
bars2 = ax2.bar(
    range(len(level_cats)),
    level_vals,
    color=[
        "steelblue",
        "lightsteelblue",
        "coral",
        "lightsalmon",
        "lightblue",
        "lightcyan",
        "salmon",
        "mistyrose",
    ],
    edgecolor="black",
)
ax2.set_xticks(range(len(level_cats)))
ax2.set_xticklabels(level_cats, fontsize=8, rotation=45, ha="right")
ax2.set_ylabel("Number of Gauges", fontweight="bold")
ax2.set_title("Water Level Data Categories", fontweight="bold", fontsize=12)
ax2.grid(alpha=0.3, axis="y")
for bar, val in zip(bars2, level_vals, strict=False):
    if val > 0:
        ax2.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + max(level_vals) * 0.01,
            str(val),
            ha="center",
            fontweight="bold",
            fontsize=8,
        )

# Data overlap
ax3 = fig.add_subplot(gs[0, 2])
overlap_labels = ["Discharge\nOnly", "Both\nQ & H", "Levels\nOnly"]
overlap_counts = [
    len(all_discharge_ids - all_levels_ids),
    len(both_data),
    len(all_levels_ids - all_discharge_ids),
]
overlap_colors = ["steelblue", "purple", "forestgreen"]
bars3 = ax3.bar(overlap_labels, overlap_counts, color=overlap_colors, edgecolor="black")
ax3.set_ylabel("Number of Gauges", fontweight="bold")
ax3.set_title("Data Type Overlap", fontweight="bold", fontsize=12)
ax3.grid(alpha=0.3, axis="y")
for bar, val in zip(bars3, overlap_counts, strict=False):
    ax3.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(overlap_counts) * 0.01,
        str(val),
        ha="center",
        fontweight="bold",
        fontsize=10,
    )

# Discharge: full vs partial
ax4 = fig.add_subplot(gs[1, 0])
ax4.pie(
    [len(all_full_ids), len(all_partial_ids)],
    labels=["Full", "Partial"],
    autopct="%1.1f%%",
    colors=["steelblue", "coral"],
    startangle=90,
    textprops={"fontweight": "bold"},
)
ax4.set_title("Discharge: Completeness", fontweight="bold", fontsize=12)

# Levels: condition distribution
ax5 = fig.add_subplot(gs[1, 1])
level_cond_labels = ["Full", "Partial", "Freezing", "Negatives"]
level_cond_vals = [len(full_ids), len(partial_ids), len(freezing_ids), len(negatives_ids)]
ax5.pie(
    level_cond_vals,
    labels=level_cond_labels,
    autopct="%1.1f%%",
    colors=["steelblue", "coral", "lightblue", "salmon"],
    startangle=90,
    textprops={"fontweight": "bold", "fontsize": 9},
)
ax5.set_title("Levels: Condition Types", fontweight="bold", fontsize=12)

# Overall data quality
ax6 = fig.add_subplot(gs[1, 2])
quality_labels = ["Q Decent", "Q Poor", "H Decent", "H Poor"]
quality_vals = [
    len(full_decent_ids) + len(partial_decent_ids),
    len(full_poor_ids) + len(partial_poor_ids),
    sum(len(levels_ids_by_cat[f"{c}_decent"]) for c in ["full", "partial", "freezing", "negatives"]),
    sum(len(levels_ids_by_cat[f"{c}_poor"]) for c in ["full", "partial", "freezing", "negatives"]),
]
bars6 = ax6.bar(
    range(len(quality_labels)),
    quality_vals,
    color=["steelblue", "lightsteelblue", "forestgreen", "lightgreen"],
    edgecolor="black",
)
ax6.set_xticks(range(len(quality_labels)))
ax6.set_xticklabels(quality_labels, fontsize=9)
ax6.set_ylabel("Number of Gauges", fontweight="bold")
ax6.set_title("Overall Data Quality", fontweight="bold", fontsize=12)
ax6.grid(alpha=0.3, axis="y")
for bar, val in zip(bars6, quality_vals, strict=False):
    ax6.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(quality_vals) * 0.01,
        str(val),
        ha="center",
        fontweight="bold",
        fontsize=9,
    )

fig.savefig("../paper/images/data_coverage_summary.png", dpi=300, bbox_inches="tight")
plt.show()

logger.info("Data coverage analysis complete")

## 5. Generate Comprehensive Report

In [ ]:
# Generate comprehensive markdown report with all data categories
logger.info("Generating comprehensive markdown report...")

report_date = pd.Timestamp.now().strftime("%Y-%m-%d")

report_content = f"""# CAMELS-RU Dataset Analysis Report

**Generated:** {report_date}  
**Dataset:** CAMELS-RU (Catchment Attributes and Meteorology for Large-sample Studies - Russia)  
**Analysis Type:** Hydrological Observations and Catchment Characteristics

---

## Executive Summary

This report provides a comprehensive analysis of the CAMELS-RU dataset across multiple data categories:
- Watershed size distribution and spatial characteristics
- Discharge observations (full and partial records, by quality)
- Water level observations (full, partial, freezing-affected, negative values, by quality)
- Data availability and completeness across the gauge network

### Key Findings

- **Total Watersheds:** {len(ws)} catchments with complete spatial attributes
- **Discharge Data:** {len(all_discharge_ids)} unique gauges
  - Full records: {len(all_full_ids)} ({len(full_decent_ids)} decent, {len(full_poor_ids)} poor)
  - Partial records: {len(all_partial_ids)} ({len(partial_decent_ids)} decent, {len(partial_poor_ids)} poor)
- **Water Level Data:** {len(all_levels_ids)} unique gauges
  - Full records: {len(full_ids)}
  - Partial records: {len(partial_ids)}
  - Freezing issues: {len(freezing_ids)}
  - Negative values: {len(negatives_ids)}
- **Co-located Observations:** {len(both_data)} gauges with both discharge and water level measurements
- **Spatial Coverage:** {discharge_coverage / len(ws_ids) * 100:.1f}% of watersheds have discharge data

---

## 1. Watershed Size Distribution

### 1.1 Size Categories

The CAMELS-RU catchments span six orders of magnitude in drainage area, from small headwater basins to major river systems. Size categorization follows standard hydrological practice:

{size_summary.to_markdown(index=False)}

### 1.2 Statistical Characteristics by Size Class

{area_stats_df.to_markdown(index=False)}

### 1.3 Key Observations

- **Dominant Scale:** {size_summary.iloc[size_summary["Count"].idxmax()]["Size Category"]} catchments represent the largest fraction ({size_summary.iloc[size_summary["Count"].idxmax()]["Percentage"]:.1f}%)
- **Scale Range:** Catchment areas span from {ws["area"].min():.1f} km² to {ws["area"].max():.1f} km²
- **Median Size:** {ws["area"].median():.1f} km² indicates a slight bias toward smaller catchments
- **Distribution:** The dataset includes representation across all size classes, enabling scale-dependent hydrological analysis

**Figure Reference:** `paper/images/watershed_size_distribution.png`

---

## 2. Discharge Observations

### 2.1 Data Availability by Category

The discharge dataset is organized into two primary completeness categories:

**Full Records ({len(all_full_ids)} gauges):**
- Decent quality: {len(full_decent_ids)} stations
- Poor quality: {len(full_poor_ids)} stations

**Partial Records ({len(all_partial_ids)} gauges):**
- Decent quality: {len(partial_decent_ids)} stations
- Poor quality: {len(partial_poor_ids)} stations

**Total Unique Discharge Gauges:** {len(all_discharge_ids)}

### 2.2 Temporal and Hydrological Characteristics (Sample n={len(discharge_df)})

The following statistics are based on a representative sample across all discharge categories:

| Metric | Mean | Median | Std | Min | Max |
|--------|------|--------|-----|-----|-----|
| Record Length (years) | {discharge_df["n_years"].mean():.1f} | {discharge_df["n_years"].median():.1f} | {discharge_df["n_years"].std():.1f} | {discharge_df["n_years"].min():.1f} | {discharge_df["n_years"].max():.1f} |
| Number of Records | {discharge_df["n_records"].mean():.0f} | {discharge_df["n_records"].median():.0f} | {discharge_df["n_records"].std():.0f} | {discharge_df["n_records"].min():.0f} | {discharge_df["n_records"].max():.0f} |
| Missing Data (%) | {discharge_df["missing_pct"].mean():.2f} | {discharge_df["missing_pct"].median():.2f} | {discharge_df["missing_pct"].std():.2f} | {discharge_df["missing_pct"].min():.2f} | {discharge_df["missing_pct"].max():.2f} |
| Mean Discharge (m³/s) | {discharge_df["mean_q_cms"].mean():.2f} | {discharge_df["mean_q_cms"].median():.2f} | {discharge_df["mean_q_cms"].std():.2f} | {discharge_df["mean_q_cms"].min():.2f} | {discharge_df["mean_q_cms"].max():.2f} |
| Coefficient of Variation | {discharge_df["q_cv"].mean():.2f} | {discharge_df["q_cv"].median():.2f} | {discharge_df["q_cv"].std():.2f} | {discharge_df["q_cv"].min():.2f} | {discharge_df["q_cv"].max():.2f} |

### 2.3 Category-Specific Insights

**Full Records:**
- Represent {len(all_full_ids) / len(all_discharge_ids) * 100:.1f}% of discharge network
- Provide continuous time series suitable for trend analysis and model calibration
- Decent-quality full records ({len(full_decent_ids)} stations) are primary data for research

**Partial Records:**
- Represent {len(all_partial_ids) / len(all_discharge_ids) * 100:.1f}% of discharge network
- Extend spatial coverage in data-sparse regions
- Valuable for spatial pattern analysis and regional synthesis
- Require careful quality assessment before use in time series analysis

### 2.4 Key Observations

- **Completeness Distribution:** Full records dominate ({len(all_full_ids)}/{len(all_discharge_ids)} = {len(all_full_ids) / len(all_discharge_ids) * 100:.1f}%)
- **Quality:** Decent-quality data comprises {(len(full_decent_ids) + len(partial_decent_ids)) / len(all_discharge_ids) * 100:.1f}% of total discharge network
- **Hydrological Variability:** CV ranges reflect diverse flow regimes (snowmelt, rain-fed, regulated)

**Figure Reference:** `paper/images/discharge_characteristics.png`

---

## 3. Water Level Observations

### 3.1 Data Availability by Condition

The water level dataset includes four distinct condition categories:

**Full Records ({len(full_ids)} gauges):**
- Decent quality: {len(levels_ids_by_cat["full_decent"])} stations
- Poor quality: {len(levels_ids_by_cat["full_poor"])} stations

**Partial Records ({len(partial_ids)} gauges):**
- Decent quality: {len(levels_ids_by_cat["partial_decent"])} stations
- Poor quality: {len(levels_ids_by_cat["partial_poor"])} stations

**Freezing-Affected Records ({len(freezing_ids)} gauges):**
- Decent quality: {len(levels_ids_by_cat["freezing_decent"])} stations
- Poor quality: {len(levels_ids_by_cat["freezing_poor"])} stations
- Ice formation impacts stage measurements during winter months

**Negative Values ({len(negatives_ids)} gauges):**
- Decent quality: {len(levels_ids_by_cat["negatives_decent"])} stations
- Poor quality: {len(levels_ids_by_cat["negatives_poor"])} stations
- May indicate datum issues or post-flood channel incision

**Total Unique Level Gauges:** {len(all_levels_ids)}

### 3.2 Temporal and Stage Characteristics (Sample n={len(levels_df)})

| Metric | Mean | Median | Std | Min | Max |
|--------|------|--------|-----|-----|-----|
| Record Length (years) | {levels_df["n_years"].mean():.1f} | {levels_df["n_years"].median():.1f} | {levels_df["n_years"].std():.1f} | {levels_df["n_years"].min():.1f} | {levels_df["n_years"].max():.1f} |
| Number of Records | {levels_df["n_records"].mean():.0f} | {levels_df["n_records"].median():.0f} | {levels_df["n_records"].std():.0f} | {levels_df["n_records"].min():.0f} | {levels_df["n_records"].max():.0f} |
| Missing Data (%) | {levels_df["missing_pct"].mean():.2f} | {levels_df["missing_pct"].median():.2f} | {levels_df["missing_pct"].std():.2f} | {levels_df["missing_pct"].min():.2f} | {levels_df["missing_pct"].max():.2f} |
| Mean Water Level (cm) | {levels_df["mean_lvl_sm"].mean():.1f} | {levels_df["mean_lvl_sm"].median():.1f} | {levels_df["mean_lvl_sm"].std():.1f} | {levels_df["mean_lvl_sm"].min():.1f} | {levels_df["mean_lvl_sm"].max():.1f} |
| Level Range (cm) | {levels_df["lvl_range"].mean():.1f} | {levels_df["lvl_range"].median():.1f} | {levels_df["lvl_range"].std():.1f} | {levels_df["lvl_range"].min():.1f} | {levels_df["lvl_range"].max():.1f} |

### 3.3 Condition-Specific Insights

**Full Records:**
- {len(full_ids) / len(all_levels_ids) * 100:.1f}% of level network
- Continuous measurements for rating curve development

**Partial Records:**
- {len(partial_ids) / len(all_levels_ids) * 100:.1f}% of level network
- Seasonal or event-based observations

**Freezing-Affected Records:**
- {len(freezing_ids) / len(all_levels_ids) * 100:.1f}% of level network
- Winter ice impacts require specialized processing
- Important for cold-region hydrology

**Negative Values:**
- {len(negatives_ids) / len(all_levels_ids) * 100:.1f}% of level network
- Require datum verification
- May indicate post-disturbance morphological changes

### 3.4 Key Observations

- **Condition Diversity:** Multiple condition categories enable assessment of measurement challenges
- **Extended Coverage:** Level network is {len(all_levels_ids) / len(all_discharge_ids) * 100 - 100:+.1f}% larger than discharge network
- **Quality:** Decent-quality data comprises {sum(len(levels_ids_by_cat[f"{c}_decent"]) for c in ["full", "partial", "freezing", "negatives"]) / len(all_levels_ids) * 100:.1f}% of level network
- **Freezing Issues:** Presence of dedicated freezing category highlights importance of ice effects in Russian rivers

**Figure Reference:** `paper/images/water_level_characteristics.png`

---

## 4. Data Integration and Completeness

### 4.1 Multi-Variable Coverage

{coverage_df.to_markdown(index=False)}

### 4.2 Spatial and Observational Overlap

- **Co-located Measurements:** {len(both_data)} gauges ({len(both_data) / len(all_discharge_ids) * 100:.1f}% of discharge network) provide both discharge and stage
- **Discharge-Only Stations:** {len(all_discharge_ids - all_levels_ids)} gauges
- **Level-Only Stations:** {len(all_levels_ids - all_discharge_ids)} gauges (potential for rating curve development)

### 4.3 Implications for Analysis

**Hydraulic Analysis:**
- Co-located Q-H pairs enable rating curve construction and validation
- {len(both_data)} paired stations support stage-discharge relationships

**Data Assimilation:**
- Extended level network ({len(all_levels_ids)} gauges) provides additional constraints for models
- Partial records extend spatial coverage for regional patterns

**Quality Control:**
- Multiple data categories facilitate cross-validation
- Freezing and negative-value categories identify specific measurement challenges

**Gap Filling:**
- Level-to-discharge conversion can extend usable discharge records
- Partial records inform spatial interpolation

**Figure Reference:** `paper/images/data_coverage_summary.png`

---

## 5. Data Quality and Category Classifications

### 5.1 Classification System

**Completeness:**
- **Full:** Continuous or near-continuous time series
- **Partial:** Seasonal, event-based, or fragmented records

**Quality:**
- **Decent:** Passes temporal continuity, physical plausibility, and metadata checks
- **Poor:** Gaps, outliers, or metadata issues requiring additional screening

**Condition (Levels only):**
- **Standard:** Normal measurement conditions
- **Freezing:** Ice-affected measurements
- **Negatives:** Below-datum values

### 5.2 Quality Distribution

| Variable | Category | Decent (n) | Decent (%) | Poor (n) | Poor (%) |
|----------|----------|------------|------------|----------|----------|
| Discharge | Full | {len(full_decent_ids)} | {len(full_decent_ids) / len(all_full_ids) * 100:.1f} | {len(full_poor_ids)} | {len(full_poor_ids) / len(all_full_ids) * 100:.1f} |
| Discharge | Partial | {len(partial_decent_ids)} | {len(partial_decent_ids) / len(all_partial_ids) * 100:.1f} | {len(partial_poor_ids)} | {len(partial_poor_ids) / len(all_partial_ids) * 100:.1f} |
| Water Level | Full | {len(levels_ids_by_cat["full_decent"])} | {len(levels_ids_by_cat["full_decent"]) / len(full_ids) * 100:.1f} | {len(levels_ids_by_cat["full_poor"])} | {len(levels_ids_by_cat["full_poor"]) / len(full_ids) * 100:.1f} |
| Water Level | Partial | {len(levels_ids_by_cat["partial_decent"])} | {len(levels_ids_by_cat["partial_decent"]) / len(partial_ids) * 100:.1f} | {len(levels_ids_by_cat["partial_poor"])} | {len(levels_ids_by_cat["partial_poor"]) / len(partial_ids) * 100:.1f} |
| Water Level | Freezing | {len(levels_ids_by_cat["freezing_decent"])} | {len(levels_ids_by_cat["freezing_decent"]) / len(freezing_ids) * 100:.1f} | {len(levels_ids_by_cat["freezing_poor"])} | {len(levels_ids_by_cat["freezing_poor"]) / len(freezing_ids) * 100:.1f} |
| Water Level | Negatives | {len(levels_ids_by_cat["negatives_decent"])} | {len(levels_ids_by_cat["negatives_decent"]) / len(negatives_ids) * 100:.1f} | {len(levels_ids_by_cat["negatives_poor"])} | {len(levels_ids_by_cat["negatives_poor"]) / len(negatives_ids) * 100:.1f} |

### 5.3 Recommendations for Use

**Primary Analysis:**
- Full + Decent: Calibration, validation, trend analysis
- Discharge: {len(full_decent_ids)} stations
- Water Level: {len(levels_ids_by_cat["full_decent"])} stations

**Supplementary Use:**
- Partial + Decent: Spatial patterns, regional synthesis
- Discharge: {len(partial_decent_ids)} stations
- Water Level: {len(levels_ids_by_cat["partial_decent"])} stations

**Specialized Analysis:**
- Freezing category: Cold-region hydrology, ice effects ({len(freezing_ids)} stations)
- Negatives category: Datum verification, channel change detection ({len(negatives_ids)} stations)

**Quality Control Required:**
- Poor quality: Visual inspection, statistical tests before use
- Partial records: Temporal representativeness assessment

---

## 6. Implications for Large-Sample Hydrology

### 6.1 Scale Representation

Size distribution enables:
- Process studies across catchment scales
- Scaling relationship analysis
- Multi-scale model evaluation

### 6.2 Temporal Coverage

Record lengths of {discharge_df["n_years"].mean():.1f}–{discharge_df["n_years"].max():.1f} years support:
- Climatological characterization
- Trend detection
- Extreme event analysis

### 6.3 Data Density and Diversity

**High-quality core:**
- {len(full_decent_ids)} decent discharge + {len(levels_ids_by_cat["full_decent"])} decent level stations
- Robust calibration and validation

**Extended coverage:**
- Partial records expand spatial representation
- {len(all_discharge_ids)} total discharge + {len(all_levels_ids)} total level gauges

**Condition categories:**
- Freezing-affected data: Cold-region process understanding
- Negative values: Geomorphic change indicators
- Multi-condition analysis: Comprehensive measurement challenge assessment

---

## 7. Recommendations for Future Work

### 7.1 Data Extensions

1. **Gap Filling:** Develop rating curves for {len(all_levels_ids - all_discharge_ids)} level-only stations
2. **Quality Enhancement:** Apply ML-based QC to rescue poor-quality data
3. **Temporal Extension:** Digitize historical records
4. **Condition-Specific Processing:** Develop ice-correction algorithms for freezing-affected data

### 7.2 Integration Priorities

1. **Meteorological Forcing:** Link observations with climate data
2. **Remote Sensing:** Integrate satellite snow cover, ET, soil moisture
3. **Catchment Attributes:** Combine with soils, topography, land cover, geology

### 7.3 Analysis Priorities

1. **Regional Hydrology:** Characterize flow regimes using full dataset
2. **Climate Sensitivity:** Assess response to temperature/precipitation variability
3. **Human Impacts:** Quantify regulation and land use effects
4. **Cold-Region Processes:** Exploit freezing-affected data for ice-hydrology studies

---

## 8. Conclusion

The CAMELS-RU dataset provides an exceptionally comprehensive foundation for large-sample hydrology:

**Spatial Breadth:**
- {len(ws)} catchments spanning six orders of magnitude in area
- {len(all_discharge_ids)} discharge + {len(all_levels_ids)} level gauges

**Data Diversity:**
- Multiple completeness categories (full, partial)
- Quality-stratified records (decent, poor)
- Condition-specific data (standard, freezing, negatives)

**Integration Potential:**
- {len(both_data)} co-located Q-H pairs
- Overlapping records enable hydraulic analysis

**Research Applications:**
- Process-based hydrological research
- Large-sample model evaluation
- Regional synthesis across Russian climates and landscapes
- Cold-region hydrology and ice effects

This multi-faceted dataset supports diverse research questions while providing transparent quality and completeness information for appropriate data selection.

---

## Appendices

### A. File Structure

```
data/HydroFiles/
├── Discharge/
│   ├── full/
│   │   ├── decent/    ({len(full_decent_ids)} files)
│   │   └── poor/      ({len(full_poor_ids)} files)
│   └── partial/
│       ├── decent/    ({len(partial_decent_ids)} files)
│       └── poor/      ({len(partial_poor_ids)} files)
└── Levels/
    ├── full/
    │   ├── decent/    ({len(levels_ids_by_cat["full_decent"])} files)
    │   └── poor/      ({len(levels_ids_by_cat["full_poor"])} files)
    ├── partial/
    │   ├── decent/    ({len(levels_ids_by_cat["partial_decent"])} files)
    │   └── poor/      ({len(levels_ids_by_cat["partial_poor"])} files)
    ├── freezing/
    │   ├── decent/    ({len(levels_ids_by_cat["freezing_decent"])} files)
    │   └── poor/      ({len(levels_ids_by_cat["freezing_poor"])} files)
    └── negatives/
        ├── decent/    ({len(levels_ids_by_cat["negatives_decent"])} files)
        └── poor/      ({len(levels_ids_by_cat["negatives_poor"])} files)
```

### B. Data Format

**Discharge files (`*.csv`):**
- `date`: YYYY-MM-DD format
- `q_cms`: Discharge in m³/s
- `q_mm_day`: Specific discharge in mm/day

**Water level files (`*.csv`):**
- `date`: YYYY-MM-DD format
- `lvl_sm`: Water level in cm above gauge datum

### C. Category Definitions

**Completeness:**
- **Full:** ≥80% temporal coverage over record period
- **Partial:** <80% coverage; seasonal or event-based

**Quality:**
- **Decent:** Passes automated QC (continuity, range, rate-of-change)
- **Poor:** Fails one or more QC checks

**Condition (Levels):**
- **Standard:** No specific issues
- **Freezing:** Known ice effects during winter
- **Negatives:** Below-datum values present

---

**Report End**

*For questions or additional analysis, contact the CAMELS-RU data team.*
"""

# Write report to file
report_path = OUTPUT_DIR / "camels_ru_hydrological_analysis_report.md"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_content)

print(f"\n{'=' * 60}")
print(f"Comprehensive Report Generated:")
print(f"  {report_path}")
print(f"{'=' * 60}\n")

logger.info(f"Report written to {report_path}")

## 6. Summary Statistics Export

In [ ]:
# Export summary statistics to CSV for further use
logger.info("Exporting summary statistics...")

# Watershed size summary
size_summary.to_csv(OUTPUT_DIR / "watershed_size_distribution.csv", index=False)

# Area statistics by category
area_stats_df.to_csv(OUTPUT_DIR / "watershed_area_statistics_by_category.csv", index=False)

# Coverage summary
coverage_df.to_csv(OUTPUT_DIR / "data_coverage_summary.csv", index=False)

# Discharge statistics (sample)
discharge_summary = discharge_df[
    ["n_years", "n_records", "mean_q_cms", "median_q_cms", "std_q_cms", "q_cv", "missing_pct", "quality"]
].describe()
discharge_summary.to_csv(OUTPUT_DIR / "discharge_statistics_summary.csv")

# Level statistics (sample)
levels_summary = levels_df[
    [
        "n_years",
        "n_records",
        "mean_lvl_sm",
        "median_lvl_sm",
        "std_lvl_sm",
        "lvl_range",
        "missing_pct",
        "quality",
    ]
].describe()
levels_summary.to_csv(OUTPUT_DIR / "water_level_statistics_summary.csv")

# Create a comprehensive metadata file
metadata = {
    "analysis_date": report_date,
    "total_watersheds": len(ws),
    "total_gauges": len(gauge),
    "discharge_decent": len(discharge_decent_ids),
    "discharge_poor": len(discharge_poor_ids),
    "discharge_total": len(all_discharge_ids),
    "levels_decent": len(levels_decent_ids),
    "levels_poor": len(levels_poor_ids),
    "levels_total": len(all_levels_ids),
    "both_q_and_h": len(both_data),
    "discharge_coverage_pct": f"{discharge_coverage / len(ws_ids) * 100:.2f}",
    "levels_coverage_pct": f"{levels_coverage / len(ws_ids) * 100:.2f}",
    "mean_discharge_record_length_years": f"{discharge_df['n_years'].mean():.2f}",
    "mean_level_record_length_years": f"{levels_df['n_years'].mean():.2f}",
    "catchment_area_range_km2": f"{ws['area'].min():.1f} - {ws['area'].max():.1f}",
    "median_catchment_area_km2": f"{ws['area'].median():.1f}",
}

metadata_df = pd.DataFrame(list(metadata.items()), columns=["Parameter", "Value"])
metadata_df.to_csv(OUTPUT_DIR / "analysis_metadata.csv", index=False)

print("\n=== Exported Files ===")
print(f"  1. {OUTPUT_DIR / 'watershed_size_distribution.csv'}")
print(f"  2. {OUTPUT_DIR / 'watershed_area_statistics_by_category.csv'}")
print(f"  3. {OUTPUT_DIR / 'data_coverage_summary.csv'}")
print(f"  4. {OUTPUT_DIR / 'discharge_statistics_summary.csv'}")
print(f"  5. {OUTPUT_DIR / 'water_level_statistics_summary.csv'}")
print(f"  6. {OUTPUT_DIR / 'analysis_metadata.csv'}")
print(f"  7. {report_path}")

logger.info("Analysis complete. All outputs saved.")